In [ ]:
# ==========================================
# 1. SETUP GOOGLE COLAB & MOUNT DRIVE
# ==========================================
from google.colab import drive
import os
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import cv2
import random
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import ResNet50
from tensorflow.keras.layers import Dense, Flatten, Dropout, Input, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils import class_weight
from collections import Counter

In [ ]:
# ==========================================
# 1. DATA LOADING (Based on your provided structure)
# ==========================================
# 1.2. Define your dataset path (Update this to your specific Drive folder)
# Example structure: /MyDrive/Dataset/Benign, /MyDrive/Dataset/Malignant, etc.
directory = '/content/drive/My Drive/Kaggle/IQ-OTH-NCCD/IQ-OTHNCCD-dataset/'

categories = ['Bengin cases', 'Malignant cases', 'Normal cases']
img_size = 224 # Optimized for ResNet50
data = []

In [ ]:
print("Loading images...")
for i in categories:
    path = os.path.join(directory, i)
    class_num = categories.index(i)
    for file in os.listdir(path):
        try:
            filepath = os.path.join(path, file)
            img = cv2.imread(filepath, 0) # Load grayscale
            img = cv2.resize(img, (img_size, img_size))
            data.append([img, class_num])
        except Exception as e:
            continue

In [ ]:
random.shuffle(data)
X, y = [], []
for feature, label in data:
    X.append(feature)
    y.append(label)

In [ ]:
X = np.array(X).reshape(-1, img_size, img_size, 1) / 255.0
y = np.array(y)

print('Data Loaded. X shape:', X.shape)
print('Class Distribution:', Counter(y))

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.20, random_state=42)

In [ ]:
# ==========================================
# 3. ADVANCED MODEL ARCHITECTURE
# ==========================================
def build_enhanced_resnet():
    # ResNet50 expects 3 channels. We use a Lambda layer to convert
    # Grayscale to RGB on the fly without increasing RAM usage.
    input_tensor = Input(shape=(img_size, img_size, 1))
    x = Lambda(lambda x: tf.image.grayscale_to_rgb(x))(input_tensor)

    base_model = ResNet50(weights='imagenet', include_top=False, input_tensor=x)

    # UNFREEZING Strategy: Unfreeze the last 10 layers (Block 5)
    # This allows the model to learn the specific high-level features of Large Cell Carcinoma.
    for layer in base_model.layers[:-10]:
        layer.trainable = False
    for layer in base_model.layers[-10:]:
        layer.trainable = True

    x = Flatten()(base_model.output)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.4)(x) # Prevents overfitting during fine-tuning
    prediction = Dense(3, activation='softmax')(x)

    return Model(inputs=input_tensor, outputs=prediction)

model = build_enhanced_resnet()

# View the structure of the model
model.summary()

# Compile the model
# model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
# ==========================================
# 4. TRAINING WITH SENSITIVITY FOCUS
# ==========================================
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), # Lower LR for fine-tuning
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
import imageio
# Image Size Variations
size_data = {}
for i in categories:
    path = os.path.join(directory, i)
    class_num = categories.index(i)
    temp_dict = {}
    for file in os.listdir(path):
        filepath = os.path.join(path, file)
        height, width, channels = imageio.imread(filepath).shape
        if str(height) + ' x ' + str(width) in temp_dict:
            temp_dict[str(height) + ' x ' + str(width)] += 1
        else:
            temp_dict[str(height) + ' x ' + str(width)] = 1

    size_data[i] = temp_dict

size_data

In [ ]:
import matplotlib.pyplot as plt
for i in categories:
    path = os.path.join(directory, i)
    class_num = categories.index(i)
    for file in os.listdir(path):
        filepath = os.path.join(path, file)
        print(i)
        img = cv2.imread(filepath, 0)
        plt.imshow(img)
        plt.show()
        break

In [ ]:
# Image Preprocessing and Testing
import matplotlib.pyplot as plt
import os

categories = ['Bengin cases', 'Malignant cases', 'Normal cases'] # Reverted to 'Bengin cases' to match existing folder

img_size = 256
for i in categories:
    cnt, samples = 0, 3
    fig, ax = plt.subplots(samples, 3, figsize=(15, 15))
    fig.suptitle(i)

    path = os.path.join(directory, i)
    class_num = categories.index(i)
    for curr_cnt, file in enumerate(os.listdir(path)):
        filepath = os.path.join(path, file)
        img = cv2.imread(filepath, 0)

        img0 = cv2.resize(img, (img_size, img_size))

        img1 = cv2.GaussianBlur(img0, (5, 5), 0)

        ax[cnt, 0].imshow(img)
        ax[cnt, 1].imshow(img0)
        ax[cnt, 2].imshow(img1)
        cnt += 1
        if cnt == samples:
            break

plt.show()

In [ ]:
callback = EarlyStopping(monitor = 'val_loss', patience = 5, restore_best_weights = True)

In [ ]:
from sklearn.utils import class_weight
from collections import Counter

# Calculate class weights for imbalanced datasets
unique_classes_in_train = np.unique(y_train)
class_weights_array = class_weight.compute_class_weight(
    'balanced',
    classes = unique_classes_in_train,
    y = y_train
)
class_weights = {i: weight for i, weight in enumerate(class_weights_array)}

history = model.fit(
    X_train, y_train,
    epochs=25,
    validation_data=(X_valid, y_valid),
    class_weight=class_weights, # Applying our sensitivity boost
    batch_size=16,
    callbacks=[callback]
)

In [ ]:
# ==========================================
# 5. EVALUATION
# ==========================================
m_model = model.predict(X_valid)
y_pred = np.argmax(m_model, axis=1)
print("\n--- Sensitivity-Focused Results ---")
print(classification_report(y_valid, y_pred, target_names=categories))

In [ ]:
y_pred = model.predict(X_valid, verbose=1)
y_pred_bool = np.argmax(y_pred, axis=1)

print(classification_report(y_valid, y_pred_bool))
print(confusion_matrix(y_true=y_valid, y_pred=y_pred_bool))

In [ ]:
# ==========================================
# 6. EVALUATION
# ==========================================
results = model.evaluate(X_valid, y_valid)
print(f"Test Loss: {results[0]}")
print(f"Test Accuracy: {results[1]}")

# Plot performance
plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label = 'val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.show()

In [ ]:
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend()
plt.show()

plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Calculate the confusion matrix
conf_mat = confusion_matrix(y_true=y_valid, y_pred=y_pred_bool)

# Plot the confusion matrix as a heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(conf_mat, annot=True, fmt='d', cmap='Blues',
            xticklabels=categories,
            yticklabels=categories)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()